In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 91.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: https://huggingface.co/meta-llama/Llama-3.1-8B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/meta-llama/Llama-3.1-8B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Load model directly
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", dtype=torch.float16,).to("cuda")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
!pip install krovetzstemmer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.9/112.9 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for krovetzstemmer: filename=KrovetzStemmer-0.8-cp312-cp312-linux_x86_64.whl size=374214 sha256=4400bfc2d38041e05b590df9aad7924731e0fc49c11e3d3e128fdcf530da3fac
  Stored in directory: /root/.cache/pip/wheels/4b/13/dc/7639b87130ee0a16dd956d81801d487ac448f25dc775437b7e
Successfully built krovetzstemmer


In [ ]:
!pip install pandas

In [ ]:
def parse_first_json_array(text: str):
    i = text.find('[')
    if i == -1:
        raise ValueError(f"No JSON array '[' found. Output was:\n{text}")

    decoder = json.JSONDecoder()
    obj, end = decoder.raw_decode(text[i:])   # parses first JSON value from there
    if not isinstance(obj, list):
        raise ValueError(f"Parsed JSON but it wasn't a list: {type(obj)}")
    return obj

In [ ]:

tag_prompt = """
    You are an ingredient phrase parser.

    You will be given an ingredient phrase.
    Tokenize by splitting on whitespace EXACTLY as given (no merging, no splitting, no re-ordering).
    You MUST output valid JSON only (no markdown, no extra text).

    Task:
    Label EACH token as exactly one of:
    - HEAD        (core food item)
    - DESCRIPTOR  (modifies/specifies the HEAD)
    - EXCLUDED    (preparation/size/quantity terms)

    HARD CONSTRAINTS (must satisfy all):
    1) There must be AT LEAST ONE token labeled "HEAD".
    2) All "HEAD" tokens must be ADJACENT to each other.
    3) Every input token must appear exactly once in the output, unchanged. Do not repeat tokens in the output!
    4) Labels must be exactly one of: "HEAD", "DESCRIPTOR", "EXCLUDED".

    Definitions:
    HEAD:
    - The core food item (a noun/food substance).
    - Can be 1+ adjacent tokens.

    DESCRIPTOR:
    - Flavor, subtype, style, origin, noun modifiers that specify the HEAD.

    EXCLUDED:
    - Prep terms (chopped, diced, minced, sliced, peeled, drained, rinsed, optional, divided, etc.)
    - Size/quantity terms (small, medium, large, extra-large, etc.)

    FALLBACK RULE (use if unsure):
    - Choose the RIGHTMOST food noun token as HEAD (or the rightmost 1-2 tokens that form a food item).
    - Prefer making a plausible food item the HEAD rather than labeling everything DESCRIPTOR.

    Output format (JSON array):
    [
    {"token": "<token>", "label": "HEAD|DESCRIPTOR|EXCLUDED"},
    ...
    ]

    Examples:

    Input: "vanilla bean ice cream"
    Output:
    [
    {"token":"vanilla","label":"DESCRIPTOR"},
    {"token":"bean","label":"DESCRIPTOR"},
    {"token":"ice","label":"HEAD"},
    {"token":"cream","label":"HEAD"}
    ]

    Input: "fresh chopped parsley"
    Output:
    [
    {"token":"fresh","label":"DESCRIPTOR"},
    {"token":"chopped","label":"EXCLUDED"},
    {"token":"parsley","label":"HEAD"}
    ]

    Input: "low sodium soy sauce"
    Output:
    [
    {"token":"low","label":"DESCRIPTOR"},
    {"token":"sodium","label":"DESCRIPTOR"},
    {"token":"soy","label":"DESCRIPTOR"},
    {"token":"sauce","label":"HEAD"}
    ]

    Now parse the next input.
    """

def tokenize_and_tag(ingredient: str):
  inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": tag_prompt}, {"role": "user", "content": ingredient}],
        tokenize=True,
        return_tensors="pt"
    ).to(model.device)

  outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False, temperature=0.0, pad_token_id=tokenizer.eos_token_id)
  input_length = inputs["input_ids"].shape[1]
  # skip past prompt tokens
  generated_tokens = outputs[0][input_length:]
  response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  return parse_first_json_array(response)


In [ ]:
best_match_prompt = """
Choose the best canonical ingredient for the given Mention.

Mention: {mention}
Head: {head_list}
Descriptors: {descriptor_list}

Candidates:
{candidate_lines}

Rules:
- Candidate must be compatible with the Head tokens.
- Prefer candidates that share descriptor tokens.
- Avoid descriptor conflicts (if both mention and candidate specify different descriptors for the same head, treat as conflict).
- A more general version is allowed (e.g., "kosher salt" -> "salt").
- If no candidate reasonably matches, set reject=true.
"""

sys_prompt = """
You are an ingredient entity linker.
You must choose the best canonical ingredient for a mention.

Follow the rules strictly.
Output format:
{"best_itemid": int|null, "reject": bool, "confidence": 0-1}
Output valid JSON only.cle
Do not include explanations outside the JSON.
"""

def get_best_match(mention: str, head_list: set[str], descriptor_list: set[str], candidates: list[tuple[int, set, str]]):
    candidate_lines = build_candidate_lines(candidates, head_list)

    prompt = best_match_prompt.format(
        mention=mention,
        head_list=sorted(head_list),
        descriptor_list=sorted(descriptor_list),
        candidate_lines=candidate_lines,
    )
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": sys_prompt}, {"role": "user", "content": prompt}],
        tokenize=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

    start = response.find("{")
    end = response.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError(f"Model did not return JSON. Got: {response}")

    response = json.loads(response[start : end + 1])
    if "best_itemid" not in response or "reject" not in response or "confidence" not in response:
        raise ValueError(f"Model did not format JSON correctly. Got: {response}")

    return response


In [ ]:
import csv
import ast
import json
from pathlib import Path
import pandas as pd
from krovetzstemmer import Stemmer
from typing import Iterable, Dict

DEBUG = True

ITEMS_OUT_PATH = "data/ingredients/foodkeeper_items.csv"
RECIPES_PATH = 'recipes-with-nutrition.csv'

MATCHED_INGREDIENTS_THRESH = 7 / 9
CHUNK_SIZE = 2_500
stemmer = Stemmer()
preparation_terms = [stemmer.stem(term) for term in ("chopped", "diced", "minced", "slice", "peeled", "drained", "rinsed", "optional", "divided")]
quantity_terms = ['large', 'big', 'small', 'medium']

RECIPES_OUT_DIR = Path("data/more_recipes/recipes")
RECIPE_ING_OUT_DIR = Path("data/more_recipes/ingredients")
RECIPE_MEALTYPE_OUT_DIR = Path("data/more_recipes/recipes_to_mealtypes")
RECIPE_CUISINE_OUT_DIR = Path("data/more_recipes/recipes_to_cuisines")
RECIPE_HEALTHLABEL_OUT_DIR = Path("data/more_recipes/healthlabels")
RECIPES_OUT_DIR.mkdir(parents=True, exist_ok=True)
RECIPE_ING_OUT_DIR.mkdir(parents=True, exist_ok=True)
RECIPE_MEALTYPE_OUT_DIR.mkdir(parents=True, exist_ok=True)
RECIPE_CUISINE_OUT_DIR.mkdir(parents=True, exist_ok=True)
RECIPE_HEALTHLABEL_OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from dataclasses import dataclass
from typing import Iterable


@dataclass(frozen=True)
class MentionParts:
    head: set[str] # main ingredient
    desc: set[str] # descriptor terms

def normalize_name_tokens(tokens_with_descriptors: list):
    """Filters EXCLUDED tokens from mention name and stems remaining tokens"""
    res = []
    for item in tokens_with_descriptors:
      token = item['token']
      label = item['label']
      if label == 'EXCLUDED':
        continue
      item['token'] = stemmer.stem(token)
      if item['token'] not in preparation_terms and item['token'] not in quantity_terms:
        res.append(item)
    return res

def extract_mention_parts(ingredient_name: str) -> MentionParts:
    """
    Uses the tokenizer/tagger for the mention ingredient-name only.
    Tokens are stemmed; EXCLUDED removed.
    """
    try:
      items = normalize_name_tokens(tokenize_and_tag(ingredient_name))
    except ValueError:
      return MentionParts(head=set(), desc=set())
    head = {it["token"] for it in items if it.get("label", "").lower() == "head"}
    desc = {it["token"] for it in items if it.get("label", "").lower() == "descriptor"}
    return MentionParts(head=head, desc=desc)


def normalize_keywords(tokens: list[str]) -> set[str]:
    res = []
    for token in tokens:
        stemmed = stemmer.stem(token)
        if stemmed not in preparation_terms and stemmed not in quantity_terms:
            res.append(stemmed)
    return set(res)


def clean_name(item_name: str) -> str:
    item_name = item_name.strip().lower()
    chars = []
    for ch in item_name:
        if ch == " " or ch.isalpha() or ch == "-":
            chars.append(ch)
    return "".join(chars)


def retrieval_score(ingredient_parts: MentionParts, kwset: set[str]) -> float:
    """
    Returns the % of mention descriptor tokens that are present in the candidate keywords set.

    Used to only rank a candidate (canonical ingredient) inside the candidate set, deterministically.
    ** Assumes that parts.head ⊆ kwset has already been checked
    """
    # Reward descriptor overlap if present
    if not ingredient_parts.desc:
        return 1.0

    overlap = len(ingredient_parts.desc & kwset)
    return overlap / max(1, len(ingredient_parts.desc))


def get_candidates(
    ingredient_name: str,
    itemid_to_keywords: dict[int, list[str]],
    *,
    max_candidates: int = 15,
    use_descriptor_filter: bool = True,
    min_desc_overlap: int = 1,
    fallback_to_head_only: bool = True,
) -> tuple[MentionParts, list[tuple[int, set, float]]]:
    """
    Returns:
      - MentionParts(head, desc)
      - candidates: list of (itemid, keywords, retrieval_score), sorted best -> worst, capped at max_candidates

    Step 1: HARD FILTER  (candidate must contain all HEAD tokens)
    Step 2: optional descriptor filter:
      keep candidates whose keywords contain >= min_desc_overlap descriptor tokens.
      BUT if it yields zero and fallback_to_head_only=True, revert to head-only.
    """
    parts = extract_mention_parts(ingredient_name)
    if not parts.head:
        return parts, []

    # First pass: HEAD-only candidates + scores
    head_only: list[tuple[int, float]] = []
    for itemid, kws in itemid_to_keywords.items():
        kwset = normalize_keywords(kws)
        if parts.head.issubset(kwset):
            head_only.append((itemid, kwset, retrieval_score(parts, kwset)))

    if not head_only:
        return parts, []

    # Optionally filter by descriptor overlap (min_desc_overlap)
    candidates = head_only
    if use_descriptor_filter and parts.desc:
        filtered: list[tuple[int, float]] = []
        for itemid, kwset, sc in head_only:
            if len(parts.desc & kwset) >= min_desc_overlap:
                filtered.append((itemid, kwset, sc))

        if filtered:
            candidates = filtered
        elif not fallback_to_head_only:
            candidates = []

    # Sort candidates by retrieval score + cap
    candidates.sort(key=lambda x: x[2], reverse=True)
    if max_candidates is not None and len(candidates) > max_candidates:
        candidates = candidates[:max_candidates]

    return parts, candidates

In [ ]:
def calc_candidates(
    ingredient_name: str,
    candidates_cache: dict[str, tuple[MentionParts, list[tuple[int, float]]]],
    itemid_to_keywords: dict[int, list[str]],
    name_to_itemid: dict[str, int],
    *,
    max_candidates: int = 60,
    use_descriptor_filter: bool = True,
    min_desc_overlap: int = 1,
) -> bool:
    """
    Cache: mention (ingredient_name) -> (MentionParts, [(itemid, keywords, score), ...])
    """
    if ingredient_name in candidates_cache:
        return True

    # Quick check to see if mention EXACTLY matches a canonical ingredient name, (but still wrap it as a singleton candidate list)
    exact = name_to_itemid.get(ingredient_name)
    if exact is not None:
        parts = extract_mention_parts(ingredient_name)
        candidates_cache[ingredient_name] = (parts, [(exact, {}, 1.0)])
        return False

    parts, candidates = get_candidates(
        ingredient_name,
        itemid_to_keywords,
        max_candidates=max_candidates,
        use_descriptor_filter=use_descriptor_filter,
        min_desc_overlap=min_desc_overlap,
        fallback_to_head_only=True,   # important for "maple syrup" -> "syrup"
    )
    candidates_cache[ingredient_name] = (parts, candidates)
    return False

In [ ]:
def write_recipe_item_links(writer: csv.writer, recipe_id: int, item_ids: Iterable[int]) -> None:
    """Writes (recipe_id, item_id) rows."""
    for item_id in item_ids:
        writer.writerow([recipe_id, item_id])

def ensure_ids_and_write_links(
    *,
    values: Iterable[str],
    to_ids: Dict[str, int],
    next_id: int,
    link_writer: csv.writer,
    link_id_start: int,
    recipe_id: int,
) -> tuple[int, int]:
    """
    For each string value:
      - ensure it has an ID in `to_ids` (assigning new IDs starting from next_id+1)
      - write a link row: [link_id, recipe_id, value_id]
    Returns: (updated_next_id, updated_link_id)
    """
    link_id = link_id_start

    for v in values:
        v_id = to_ids.get(v)
        if v_id is None:
            next_id += 1
            v_id = next_id
            to_ids[v] = v_id

        link_writer.writerow([link_id, recipe_id, v_id])
        link_id += 1

    return next_id, link_id

def safe_literal_list(s: str) -> list:
    if s is None:
        return []
    s = str(s).strip()
    if not s:
        return []
    try:
        val = ast.literal_eval(s)
        return val if isinstance(val, list) else []
    except (ValueError, SyntaxError):
        return []

def get_mapping(ingredients_df: pd.DataFrame):
    itemid_to_keywords: dict[int, list[str]] = {}
    itemid_to_name: dict[int, str] = {}
    name_to_itemid: dict[str, int] = {}

    for _, row in ingredients_df.iterrows():
        itemid = int(row["id"])
        name = str(row["name"]).strip().lower()

        kws = row["keywords"]
        keywords = ast.literal_eval(kws) if isinstance(kws, str) else list(kws)
        keywords = [str(k).strip().lower() for k in keywords if str(k).strip()]

        itemid_to_name[itemid] = name
        itemid_to_keywords[itemid] = keywords
        name_to_itemid[name] = itemid

    return itemid_to_keywords, itemid_to_name, name_to_itemid

def open_chunk_writers(idx: int):
    recipes_f = open(RECIPES_OUT_DIR / f"{idx}.csv", "w", newline="", encoding="utf-8")
    ing_f = open(RECIPE_ING_OUT_DIR / f"{idx}.csv", "w", newline="", encoding="utf-8")
    recipe_mealtype_f = open(RECIPE_MEALTYPE_OUT_DIR / f"{idx}.csv", "w", newline="", encoding="utf-8")
    recipe_cuisine_f = open(RECIPE_CUISINE_OUT_DIR / f"{idx}.csv", "w", newline="", encoding="utf-8")
    recipe_healthlabel_f = open(RECIPE_HEALTHLABEL_OUT_DIR / f"{idx}.csv", "w", newline="", encoding="utf-8")

    recipes_w = csv.writer(recipes_f)
    ing_w = csv.writer(ing_f)
    recipe_meals_w = csv.writer(recipe_mealtype_f)
    recipe_cuisine_w = csv.writer(recipe_cuisine_f)
    recipe_healthlabel_w = csv.writer(recipe_healthlabel_f)

    recipes_w.writerow(["RecipeID", "RecipeName", "Link", "Image_Link"])
    ing_w.writerow(["RecipeID", "IngredientID"])
    recipe_meals_w.writerow(["ID", "RecipeID", "MealTypeID"])
    recipe_cuisine_w.writerow(["ID", "RecipeID", "CuisineID"])
    recipe_healthlabel_w.writerow(["ID", "RecipeID", "HealthLabelID"])

    return (
        recipes_f, recipes_w,
        ing_f, ing_w,
        recipe_mealtype_f, recipe_meals_w,
        recipe_cuisine_f, recipe_cuisine_w,
        recipe_healthlabel_f, recipe_healthlabel_w
    )

In [ ]:
def build_candidate_lines(
    candidates: list[tuple[int, str, list[str]]],  # (itemid, name, keywords)
    mention_head: set[str],
    *,
    max_kw: int = 20,
) -> str:
    lines = []
    for itemid, name, kws in candidates:
        filtered = [k for k in kws if k not in mention_head]

        # cap max number of keywords to keep prompt small
        filtered = filtered[:max_kw]

        kw_str = ", ".join(filtered) if filtered else ""
        lines.append(f"{itemid}: {name} | keywords: [{kw_str}]")

    return "\n".join(lines)

In [ ]:
if __name__ == "__main__":
    # cache to store ingredient scores
    candidates_cache: dict[str, tuple[MentionParts, list[tuple[int, set, float]]]] = {}
    candidates_matches: dict[str, int] = {}
    ingredients_df = pd.read_csv(ITEMS_OUT_PATH)
    itemid_to_keywords, itemid_to_name, name_to_itemid = get_mapping(ingredients_df)

    chunk_index = 0
    row_in_chunk = 0
    recipe_index = 0
    num_recipes_created = 0

    (
        recipes_f, recipes_w,
        ing_f, ing_w,
        recipe_mealtype_f, recipe_meals_w,
        recipe_cuisine_f, recipe_cuisine_w,
        recipe_healthlabel_f, recipe_healthlabel_w
    ) = open_chunk_writers(chunk_index)

    meals_to_ids = {}
    cuisines_to_ids = {}
    healthlabels_to_ids = {}
    next_meal_id, next_cuisine_id, next_healthlabel_id = 0, 0, 0
    rm_id, rc_id, rh_id = 0, 0, 0

    with open(RECIPES_PATH, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            title = row.get("recipe_name", "")
            link = row.get("url", "")
            image_link = row.get("image_url", "")
            ner_terms = [food_info['food'] for food_info in safe_literal_list(row.get("ingredients", ""))]
            if not ner_terms:
                continue

            meal_types = [meal_type.lower() for meal_type in safe_literal_list(row['meal_type'])[0].split('/')]
            cuisine_types = [cuisine.lower() for cuisine in safe_literal_list(row['cuisine_type'])]
            health_labels = [health_label.lower() for health_label in safe_literal_list(row['health_labels'])]

            matched = []
            for ing_name in ner_terms:
                ing_name = clean_name(ing_name)
                ing_seen = calc_candidates(
                    ing_name,
                    candidates_cache,
                    itemid_to_keywords,
                    name_to_itemid,
                    max_candidates=15,
                    use_descriptor_filter=True,
                    min_desc_overlap=1,
                )
                if ing_seen and ing_name in candidates_matches:
                  matched.append(candidates_matches[ing_name])
                  continue

                parts, candidates = candidates_cache[ing_name]
                if not candidates:
                    #print('no candidates')
                    continue

                candidates = [(itemid, kws, itemid_to_name[itemid]) for itemid, kws, _ in candidates]
                best_canonical_match = get_best_match(ing_name, parts.head, parts.desc, candidates)
                #print(ing_name, best_canonical_match['best_itemid'], best_canonical_match['confidence'])
                if best_canonical_match['reject']:
                    continue
                elif best_canonical_match['best_itemid'] and best_canonical_match['confidence'] >= 0.80:
                    matched.append(best_canonical_match['best_itemid'])
                    candidates_matches[ing_name] = best_canonical_match['best_itemid']


            if len(matched) / len(ner_terms) < MATCHED_INGREDIENTS_THRESH:
                continue

            recipe_id = recipe_index
            recipe_index += 1
            #print(recipe_index)
            num_recipes_created += 1

            recipes_w.writerow([recipe_id, title, link, image_link])
            write_recipe_item_links(ing_w, recipe_id, matched)

            next_meal_id, rm_id = ensure_ids_and_write_links(
                values=meal_types,
                to_ids=meals_to_ids,
                next_id=next_meal_id,
                link_writer=recipe_meals_w,
                link_id_start=rm_id,
                recipe_id=recipe_id,
            )
            next_cuisine_id, rc_id = ensure_ids_and_write_links(
                values=cuisine_types,
                to_ids=cuisines_to_ids,
                next_id=next_cuisine_id,
                link_writer=recipe_cuisine_w,
                link_id_start=rc_id,
                recipe_id=recipe_id,
            )
            next_healthlabel_id, rh_id = ensure_ids_and_write_links(
                values=health_labels,
                to_ids=healthlabels_to_ids,
                next_id=next_healthlabel_id,
                link_writer=recipe_healthlabel_w,
                link_id_start=rh_id,
                recipe_id=recipe_id,
            )

            row_in_chunk += 1
            if row_in_chunk >= CHUNK_SIZE:
                recipes_f.close()
                ing_f.close()
                recipe_mealtype_f.close()
                recipe_cuisine_f.close()
                recipe_healthlabel_f.close()

                chunk_index += 1
                row_in_chunk = 0
                (  recipes_f, recipes_w,
                    ing_f, ing_w,
                    recipe_mealtype_f, recipe_meals_w,
                    recipe_cuisine_f, recipe_cuisine_w,
                    recipe_healthlabel_f, recipe_healthlabel_w
                ) = open_chunk_writers(chunk_index)

    recipes_f.close()
    ing_f.close()
    recipe_mealtype_f.close()
    recipe_cuisine_f.close()
    recipe_healthlabel_f.close()

    with open('meals_to_ids', 'w') as f:
        json.dump(meals_to_ids, f)
    with open('cuisines_to_ids', 'w') as f:
        json.dump(cuisines_to_ids, f)
    with open('health_labels_to_ids', 'w') as f:
        json.dump(healthlabels_to_ids, f)

    if DEBUG:
        # with open("similarity_scores.json", "w", encoding="utf-8") as out:
        #     json.dump(ingredients_cache, out, indent=2)
        print(f"Total number of recipes successfully created: {num_recipes_created}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

In [ ]:
print(healthlabels_to_ids)

{'vegetarian': 1, 'gluten-free': 2, 'peanut-free': 3, 'tree-nut-free': 4, 'soy-free': 5, 'fish-free': 6, 'shellfish-free': 7, 'vegan': 8, 'dairy-free': 9, 'egg-free': 10, 'paleo': 11, 'low sugar': 12}


In [ ]:
print(cuisines_to_ids)

{'american': 1, 'asian': 2, 'italian': 3, 'nordic': 4, 'mediterranean': 5, 'british': 6, 'chinese': 7, 'eastern europe': 8, 'french': 9, 'world': 10, 'middle eastern': 11, 'indian': 12, 'mexican': 13, 'south east asian': 14, 'south american': 15, 'japanese': 16, 'central europe': 17, 'greek': 18, 'caribbean': 19, 'korean': 20, 'kosher': 21}


In [ ]:
print(num_recipes_created)

7542


In [ ]:
import shutil

shutil.rmtree("data/more_recipes")

In [ ]:
with open('meals_to_ids', 'w') as f:
        json.dump(meals_to_ids, f)
with open('cuisines_to_ids', 'w') as f:
    json.dump(cuisines_to_ids, f)
with open('health_labels_to_ids', 'w') as f:
    json.dump(healthlabels_to_ids, f)

In [ ]:
from google.colab import files
files.download('meals_to_ids')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
import shutil

# pickle.dump(candidates_cache, open('candidates_cache.pkl', 'wb'))
# files.download('candidates_cache.pkl')

# pickle.dump(candidates_matches, open('candidates_matches.pkl', 'wb'))
# files.download('candidates_matches.pkl')

shutil.make_archive('more_recipes', 'zip', 'data/more_recipes')
files.download('more_recipes.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>